# Category 6 — Window Functions
### Total Questions: 8

---

1. Employee with maximum salary per department (window)
2. Second highest salary
3. Name of employee with second highest salary
4. Second highest salary per department
5. Rank employees by salary per department
6. Rank users based on total amount spent
7. emp_id and department for recently joined employee per dept
8. Highest salary in each dept (employees + department tables)

Upload sql_practice.db from your computer

In [ ]:
# ⬆️ Upload sql_practice.db from your computer
from google.colab import files
uploaded = files.upload()
print("✅ Database uploaded!")

Saving sql_practice.db to sql_practice.db
✅ Database uploaded!


Verify tables loaded

In [ ]:
import sqlite3
import pandas as pd
import os

_cwd = os.getcwd()
_db = os.path.join(_cwd, "sql_practice.db")
if not os.path.isfile(_db):
    _db = os.path.join(os.path.dirname(_cwd), "sql_practice.db")
conn = sqlite3.connect(_db)

# Verify tables loaded
pd.read_sql_query("""
SELECT name FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)


,name
0,customers
1,department
2,emp
3,employee_salary
4,employees
5,monthly_salary
6,num
7,orders
8,posts
9,products


# Q1. Employee With Maximum Salary Per Department (Window)

## 1. Employee With Maximum Salary Per Department

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / MAX OVER

---

**Table: employee_salary**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| did              | int     |
| name_of_employee | varchar |
| salary           | int     |
+------------------+---------+
```

- `did` represents the department ID.
- `name_of_employee` represents the employee's name.
- `salary` represents the employee's salary.

Each department may contain multiple employees.

---

**Problem**

Write a SQL query to return the **name of the employee with the maximum salary** in each department using a window function.

Return `did`, `name_of_employee` and `salary`.

**Example 1**

**Input**

employee_salary table:
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Alice            | 50000  |
| 1   | Bob              | 65000  |
| 1   | Carol            | 60000  |
| 2   | David            | 70000  |
| 2   | Emma             | 72000  |
| 2   | Frank            | 68000  |
| 3   | Anna             | 55000  |
| 3   | George           | 52000  |
| 3   | Henry            | 51000  |
+-----+------------------+--------+
```

**Output**
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Bob              | 65000  |
| 2   | Emma             | 72000  |
| 3   | Anna             | 55000  |
+-----+------------------+--------+
```

CODE (View Table):

In [ ]:
# 👀 Preview the table
print("employee_salary table:")
pd.read_sql_query("SELECT * FROM employee_salary", conn)

employee_salary table:


,did,name_of_employee,salary
0,1,Alice,50000
1,1,Bob,65000
2,1,Carol,60000
3,2,David,70000
4,2,Emma,72000
5,2,Frank,68000
6,3,Anna,55000
7,3,George,52000
8,3,Henry,51000


CODE (Solution):

In [ ]:
# ✅ Solution
query = """
SELECT did,
       name_of_employee,
       salary
FROM (
    SELECT did,
           name_of_employee,
           salary,
           RANK() OVER (PARTITION BY did ORDER BY salary DESC) AS rnk
    FROM employee_salary
)
WHERE rnk = 1
"""

pd.read_sql_query(query, conn)

,did,name_of_employee,salary
0,1,Bob,65000
1,2,Emma,72000
2,3,Anna,55000


**Explanation**

- `PARTITION BY did` divides the data into groups by department.
- `ORDER BY salary DESC` ranks employees within each department from highest to lowest salary.
- `RANK() = 1` gives us the employee with the **highest salary** in each department.
- We wrap it in a subquery because we cannot use window functions directly in a `WHERE` clause.

# Q2. Second Highest Salary

2 Second Highest Salary

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / DENSE_RANK

---

**Table: employee_salary**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| did              | int     |
| name_of_employee | varchar |
| salary           | int     |
+------------------+---------+
```

- `did` represents the department ID.
- `name_of_employee` represents the employee's name.
- `salary` represents the employee's salary.

---

**Problem**

Write a SQL query to return the **second highest salary** from the entire table.

**Example 1**

**Input**

employee_salary table:
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Alice            | 50000  |
| 1   | Bob              | 65000  |
| 1   | Carol            | 60000  |
| 2   | David            | 70000  |
| 2   | Emma             | 72000  |
| 2   | Frank            | 68000  |
| 3   | Anna             | 55000  |
| 3   | George           | 52000  |
| 3   | Henry            | 51000  |
+-----+------------------+--------+
```

**Output**
```
+--------+
| salary |
+--------+
| 70000  |
+--------+
```

 CODE (View Table):

In [ ]:
# 👀 Preview the table
print("employee_salary table:")
pd.read_sql_query("SELECT * FROM employee_salary", conn)

employee_salary table:


,did,name_of_employee,salary
0,1,Alice,50000
1,1,Bob,65000
2,1,Carol,60000
3,2,David,70000
4,2,Emma,72000
5,2,Frank,68000
6,3,Anna,55000
7,3,George,52000
8,3,Henry,51000


 CODE (Solution):

In [ ]:
# ✅ Solution
query = """
SELECT salary
FROM (
    SELECT salary,
           DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM employee_salary
)
WHERE rnk = 2
LIMIT 1
"""

pd.read_sql_query(query, conn)

,salary
0,70000


**Explanation**

- `DENSE_RANK()` assigns ranks without gaps — if two employees share rank 1, the next rank is still 2 (not 3).
- `ORDER BY salary DESC` ranks salaries from highest to lowest.
- `WHERE rnk = 2` picks the second highest salary.
- We use `DENSE_RANK` instead of `RANK` to correctly handle duplicate salaries.

# Q3. Name of Employee With Second Highest Salary



**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / DENSE_RANK

---

**Table: employee_salary**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| did              | int     |
| name_of_employee | varchar |
| salary           | int     |
+------------------+---------+
```

- `did` represents the department ID.
- `name_of_employee` represents the employee's name.
- `salary` represents the employee's salary.

---

**Problem**

Write a SQL query to return the **name of the employee with the second highest salary** from the entire table.

**Example 1**

**Input**

employee_salary table:
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Alice            | 50000  |
| 1   | Bob              | 65000  |
| 1   | Carol            | 60000  |
| 2   | David            | 70000  |
| 2   | Emma             | 72000  |
| 2   | Frank            | 68000  |
| 3   | Anna             | 55000  |
| 3   | George           | 52000  |
| 3   | Henry            | 51000  |
+-----+------------------+--------+
```

**Output**
```
+------------------+--------+
| name_of_employee | salary |
+------------------+--------+
| David            | 70000  |
+------------------+--------+
```

CODE (View Table):

In [ ]:
# 👀 Preview the table
print("employee_salary table:")
pd.read_sql_query("SELECT * FROM employee_salary", conn)

employee_salary table:


,did,name_of_employee,salary
0,1,Alice,50000
1,1,Bob,65000
2,1,Carol,60000
3,2,David,70000
4,2,Emma,72000
5,2,Frank,68000
6,3,Anna,55000
7,3,George,52000
8,3,Henry,51000


 CODE (Solution):

In [ ]:
# ✅ Solution
query = """
SELECT name_of_employee,
       salary
FROM (
    SELECT name_of_employee,
           salary,
           DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM employee_salary
)
WHERE rnk = 2
"""

pd.read_sql_query(query, conn)

,name_of_employee,salary
0,David,70000


**Explanation**

- `DENSE_RANK() OVER (ORDER BY salary DESC)` ranks all employees globally by salary.
- `WHERE rnk = 2` filters only the employee(s) with the **second highest salary**.
- We include `name_of_employee` in the SELECT to return the name along with the salary.
- If multiple employees share the second highest salary, all of them will be returned.

# Q4. Second Highest Salary Per Department



**Difficulty:** 🔴 Hard

**Subtopic:** Window Functions / DENSE_RANK + PARTITION BY

---

**Table: employee_salary**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| did              | int     |
| name_of_employee | varchar |
| salary           | int     |
+------------------+---------+
```

- `did` represents the department ID.
- `name_of_employee` represents the employee's name.
- `salary` represents the employee's salary.

---

**Problem**

Write a SQL query to return the **second highest salary within each department**.

Return `did`, `name_of_employee` and `salary`.

**Example 1**

**Input**

employee_salary table:
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Alice            | 50000  |
| 1   | Bob              | 65000  |
| 1   | Carol            | 60000  |
| 2   | David            | 70000  |
| 2   | Emma             | 72000  |
| 2   | Frank            | 68000  |
| 3   | Anna             | 55000  |
| 3   | George           | 52000  |
| 3   | Henry            | 51000  |
+-----+------------------+--------+
```

**Output**
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Carol            | 60000  |
| 2   | David            | 70000  |
| 3   | George           | 52000  |
+-----+------------------+--------+
```

Preview the table

In [ ]:
# 👀 Preview the table
print("employee_salary table:")
pd.read_sql_query("SELECT * FROM employee_salary", conn)

employee_salary table:


,did,name_of_employee,salary
0,1,Alice,50000
1,1,Bob,65000
2,1,Carol,60000
3,2,David,70000
4,2,Emma,72000
5,2,Frank,68000
6,3,Anna,55000
7,3,George,52000
8,3,Henry,51000


✅ Solution

In [ ]:
# ✅ Solution
query = """
SELECT did,
       name_of_employee,
       salary
FROM (
    SELECT did,
           name_of_employee,
           salary,
           DENSE_RANK() OVER (PARTITION BY did ORDER BY salary DESC) AS rnk
    FROM employee_salary
)
WHERE rnk = 2
"""

pd.read_sql_query(query, conn)

,did,name_of_employee,salary
0,1,Carol,60000
1,2,David,70000
2,3,George,52000


**Explanation**

- `PARTITION BY did` resets the ranking for each department separately.
- `ORDER BY salary DESC` ranks employees from highest to lowest within each department.
- `DENSE_RANK() = 2` picks the employee with the **second highest salary** in each department.
- The difference from Q2 — here we add `PARTITION BY did` so ranking happens **per department** not globally.

## 5. Rank Employees by Salary Per Department

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / RANK + PARTITION BY

---

**Table: employee_salary**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| did              | int     |
| name_of_employee | varchar |
| salary           | int     |
+------------------+---------+
```

- `did` represents the department ID.
- `name_of_employee` represents the employee's name.
- `salary` represents the employee's salary.

---

**Problem**

Write a SQL query to **rank employees by their salary within each department**.

Return `did`, `name_of_employee`, `salary` and `rank`.

**Example 1**

**Input**

employee_salary table:
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Alice            | 50000  |
| 1   | Bob              | 65000  |
| 1   | Carol            | 60000  |
| 2   | David            | 70000  |
| 2   | Emma             | 72000  |
| 2   | Frank            | 68000  |
| 3   | Anna             | 55000  |
| 3   | George           | 52000  |
| 3   | Henry            | 51000  |
+-----+------------------+--------+
```

**Output**
```
+-----+------------------+--------+------+
| did | name_of_employee | salary | rank |
+-----+------------------+--------+------+
| 1   | Bob              | 65000  | 1    |
| 1   | Carol            | 60000  | 2    |
| 1   | Alice            | 50000  | 3    |
| 2   | Emma             | 72000  | 1    |
| 2   | David            | 70000  | 2    |
| 2   | Frank            | 68000  | 3    |
| 3   | Anna             | 55000  | 1    |
| 3   | George           | 52000  | 2    |
| 3   | Henry            | 51000  | 3    |
+-----+------------------+--------+------+
```

CODE (View Table):

Preview the table

In [ ]:
# 👀 Preview the table
print("employee_salary table:")
pd.read_sql_query("SELECT * FROM employee_salary", conn)

employee_salary table:


,did,name_of_employee,salary
0,1,Alice,50000
1,1,Bob,65000
2,1,Carol,60000
3,2,David,70000
4,2,Emma,72000
5,2,Frank,68000
6,3,Anna,55000
7,3,George,52000
8,3,Henry,51000


CODE (Solution):

In [ ]:
# ✅ Solution
query = """
SELECT did,
       name_of_employee,
       salary,
       RANK() OVER (PARTITION BY did ORDER BY salary DESC) AS rank
FROM employee_salary
"""

pd.read_sql_query(query, conn)

,did,name_of_employee,salary,rank
0,1,Bob,65000,1
1,1,Carol,60000,2
2,1,Alice,50000,3
3,2,Emma,72000,1
4,2,David,70000,2
5,2,Frank,68000,3
6,3,Anna,55000,1
7,3,George,52000,2
8,3,Henry,51000,3


**Explanation**

- `PARTITION BY did` divides employees into groups by department.
- `ORDER BY salary DESC` orders employees from highest to lowest salary within each group.
- `RANK()` assigns a rank to each employee — if two employees have the same salary they get the same rank and the next rank is skipped.
- Unlike Q2 and Q3 we do not filter here — we return **all employees** with their rank.

## 6. Rank Users Based on Total Amount Spent

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / RANK + GROUP BY

---

**Table: zomato_orders**
```
+---------------------+---------+
| Column Name         | Type    |
+---------------------+---------+
| user_id             | int     |
| order_id            | int     |
| amount              | int     |
| order_date          | date    |
| delivery_partner_id | int     |
| delivery_rating     | int     |
+---------------------+---------+
```

- `user_id` represents the user who placed the order.
- `amount` represents the order amount.

---

**Problem**

Write a SQL query to **rank users based on their total amount spent** on Zomato.

The user who spent the most should have rank 1.

Return `user_id`, `total_amount` and `rank`.

**Example 1**

**Input**

zomato_orders table:
```
+---------+----------+--------+------------+---------------------+-----------------+
| user_id | order_id | amount | order_date | delivery_partner_id | delivery_rating |
+---------+----------+--------+------------+---------------------+-----------------+
| 1       | 101      | 200    | 2024-01-01 | 1                   | 5               |
| 1       | 102      | 300    | 2024-01-03 | 2                   | 4               |
| 2       | 103      | 150    | 2024-01-04 | 1                   | 5               |
| 3       | 104      | 400    | 2024-01-05 | 3                   | 3               |
+---------+----------+--------+------------+---------------------+-----------------+
```

**Output**
```
+---------+--------------+------+
| user_id | total_amount | rank |
+---------+--------------+------+
| 1       | 500          | 1    |
| 3       | 400          | 2    |
| 2       | 150          | 3    |
+---------+--------------+------+
```

Preview the table

In [ ]:
# 👀 Preview the table
print("zomato_orders table:")
pd.read_sql_query("SELECT * FROM zomato_orders", conn)

zomato_orders table:


,user_id,order_id,amount,order_date,delivery_partner_id,delivery_rating
0,1,101,200,2024-01-01,1,5
1,1,102,300,2024-01-03,2,4
2,2,103,150,2024-01-04,1,5
3,3,104,400,2024-01-05,3,3


In [ ]:
# ✅ Solution
query = """
SELECT user_id,
       total_amount,
       RANK() OVER (ORDER BY total_amount DESC) AS rank
FROM (
    SELECT user_id,
           SUM(amount) AS total_amount
    FROM zomato_orders
    GROUP BY user_id
)
"""

pd.read_sql_query(query, conn)

,user_id,total_amount,rank
0,1,500,1
1,3,400,2
2,2,150,3


**Explanation**

- The **inner subquery** first calculates `SUM(amount)` per user using `GROUP BY user_id`.
- The **outer query** applies `RANK() OVER (ORDER BY total_amount DESC)` to rank users from highest to lowest spender.
- No `PARTITION BY` is needed here because we are ranking **all users globally** — not within groups.
- User 1 spent 500 total (200+300) so they get rank 1.

## 7. Most Recently Joined Employee Per Department

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / RANK + PARTITION BY + DATE

---

**Tables: employees + department**
```
employees
+------------+---------+
| Column Name| Type    |
+------------+---------+
| id         | int     |
| name       | varchar |
| salary     | int     |
| dep_id     | int     |
| manager_id | int     |
| hire_date  | date    |
+------------+---------+

department
+------------+---------+
| Column Name| Type    |
+------------+---------+
| id         | int     |
| name       | varchar |
+------------+---------+
```

- `hire_date` represents when the employee joined.
- `dep_id` links to `id` in the department table.

---

**Problem**

Write a SQL query to get the **emp_id and department** for the employee who **most recently joined** in each department and is still working.

**Example 1**

**Input**

employees table:
```
+----+-------+--------+--------+------------+------------+
| id | name  | salary | dep_id | manager_id | hire_date  |
+----+-------+--------+--------+------------+------------+
| 1  | John  | 60000  | 1      | NULL       | 2023-01-01 |
| 2  | Alice | 80000  | 1      | 1          | 2023-03-01 |
| 3  | Bob   | 75000  | 2      | 1          | 2023-05-01 |
| 4  | David | 50000  | 2      | 3          | 2024-01-01 |
+----+-------+--------+--------+------------+------------+
```

department table:
```
+----+---------+
| id | name    |
+----+---------+
| 1  | IT      |
| 2  | Finance |
| 3  | HR      |
+----+---------+
```

**Output**
```
+----+------+---------+------------+
| id | name | dept    | hire_date  |
+----+------+---------+------------+
| 2  | Alice| IT      | 2023-03-01 |
| 4  | David| Finance | 2024-01-01 |
+----+------+---------+------------+
```

CODE (View Tables):

Preview Table 1

In [ ]:
# 👀 Preview Table 1
print("employees table:")
pd.read_sql_query("SELECT * FROM employees", conn)

employees table:


,id,name,salary,dep_id,manager_id,hire_date,email
0,1,John,60000,1,NaN,2026-01-15,john@gmail.com
1,2,Alice,80000,1,1.0,2026-02-01,alice@gmail.com
2,3,Bob,75000,2,1.0,2026-01-20,john@gmail.com
3,4,David,50000,2,3.0,2026-03-01,bob@gmail.com
4,5,John,45000,3,3.0,2026-02-15,david@gmail.com
5,6,Emma,90000,1,1.0,2025-06-10,emma@gmail.com
6,7,Frank,55000,2,3.0,2025-08-20,frank@gmail.com
7,8,Grace,72000,3,5.0,2026-01-05,grace@gmail.com
8,9,Henry,48000,3,5.0,2025-11-11,alice@gmail.com
9,10,Isla,95000,1,1.0,2025-04-01,isla@gmail.com


In [ ]:
# 👀 Preview Table 2
print("department table:")
pd.read_sql_query("SELECT * FROM department", conn)

department table:


,id,name
0,1,IT
1,2,Finance
2,3,HR


CODE (Solution):

In [ ]:
# ✅ Solution
query = """
SELECT id,
       name,
       dept,
       hire_date
FROM (
    SELECT e.id,
           e.name,
           d.name AS dept,
           e.hire_date,
           RANK() OVER (PARTITION BY e.dep_id ORDER BY e.hire_date DESC) AS rnk
    FROM employees e
    JOIN department d ON e.dep_id = d.id
)
WHERE rnk = 1
"""

pd.read_sql_query(query, conn)

,id,name,dept,hire_date
0,18,Quinn,IT,2026-02-10
1,16,Olivia,Finance,2026-03-05
2,5,John,HR,2026-02-15


**Explanation**

- `JOIN department d ON e.dep_id = d.id` connects employees to their department name.
- `PARTITION BY e.dep_id` resets the rank for each department.
- `ORDER BY e.hire_date DESC` ranks employees from most recently joined to oldest within each department.
- `WHERE rnk = 1` keeps only the **most recently joined** employee in each department.

## 8. Highest Salary in Each Department

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / RANK + JOIN

---

**Tables: employees + department**
```
employees
+------------+---------+
| Column Name| Type    |
+------------+---------+
| id         | int     |
| name       | varchar |
| salary     | int     |
| dep_id     | int     |
| manager_id | int     |
| hire_date  | date    |
+------------+---------+

department
+------------+---------+
| Column Name| Type    |
+------------+---------+
| id         | int     |
| name       | varchar |
+------------+---------+
```

- `dep_id` in employees links to `id` in department.
- `salary` represents the employee's salary.

---

**Problem**

Write a SQL query to return the **name of the employee with the highest salary** in each department using the employees and department tables.

Return `employee name`, `department name` and `salary`.

**Example 1**

**Input**

employees table:
```
+----+-------+--------+--------+------------+------------+
| id | name  | salary | dep_id | manager_id | hire_date  |
+----+-------+--------+--------+------------+------------+
| 1  | John  | 60000  | 1      | NULL       | 2023-01-01 |
| 2  | Alice | 80000  | 1      | 1          | 2023-03-01 |
| 3  | Bob   | 75000  | 2      | 1          | 2023-05-01 |
| 4  | David | 50000  | 2      | 3          | 2024-01-01 |
+----+-------+--------+--------+------------+------------+
```

department table:
```
+----+---------+
| id | name    |
+----+---------+
| 1  | IT      |
| 2  | Finance |
| 3  | HR      |
+----+---------+
```

**Output**
```
+-------+---------+--------+
| name  | dept    | salary |
+-------+---------+--------+
| Alice | IT      | 80000  |
| Bob   | Finance | 75000  |
+-------+---------+--------+
```

 CODE (View Tables):

In [ ]:
# 👀 Preview Table 1
print("employees table:")
pd.read_sql_query("SELECT * FROM employees", conn)

employees table:


,id,name,salary,dep_id,manager_id,hire_date,email
0,1,John,60000,1,NaN,2026-01-15,john@gmail.com
1,2,Alice,80000,1,1.0,2026-02-01,alice@gmail.com
2,3,Bob,75000,2,1.0,2026-01-20,john@gmail.com
3,4,David,50000,2,3.0,2026-03-01,bob@gmail.com
4,5,John,45000,3,3.0,2026-02-15,david@gmail.com
5,6,Emma,90000,1,1.0,2025-06-10,emma@gmail.com
6,7,Frank,55000,2,3.0,2025-08-20,frank@gmail.com
7,8,Grace,72000,3,5.0,2026-01-05,grace@gmail.com
8,9,Henry,48000,3,5.0,2025-11-11,alice@gmail.com
9,10,Isla,95000,1,1.0,2025-04-01,isla@gmail.com


In [ ]:
# 👀 Preview Table 2
print("department table:")
pd.read_sql_query("SELECT * FROM department", conn)

department table:


,id,name
0,1,IT
1,2,Finance
2,3,HR


In [ ]:
# ✅ Solution
query = """
SELECT name,
       dept,
       salary
FROM (
    SELECT e.name,
           d.name AS dept,
           e.salary,
           RANK() OVER (PARTITION BY e.dep_id ORDER BY e.salary DESC) AS rnk
    FROM employees e
    JOIN department d ON e.dep_id = d.id
)
WHERE rnk = 1
"""

pd.read_sql_query(query, conn)

,name,dept,salary
0,Isla,IT,95000
1,Rachel,Finance,79000
2,Paul,HR,88000


**Explanation**

- `JOIN department d ON e.dep_id = d.id` fetches the department name for each employee.
- `PARTITION BY e.dep_id` resets the ranking for each department separately.
- `ORDER BY e.salary DESC` ranks employees from highest to lowest salary within each department.
- `WHERE rnk = 1` returns only the **top earning employee** in each department.
- This is similar to Cat 2 Q1 but uses a JOIN to show the department name instead of just the ID.